# Respondendo às perguntas de negócio

## 1. O ranking de entrada prediz o resultado do campeonato? Com que frequência a dupla pior ranqueada vence, e isso muda entre grupos, qualificatória e eliminatória?

In [0]:
%sql
-- Zebra: a dupla de seed pior venceu. Seed principal quando as duas têm; da qualificatória quando só têm esse.
WITH base AS (
  SELECT f.fase,
         CASE WHEN p.w_seed_principal IS NOT NULL AND p.l_seed_principal IS NOT NULL
                THEN p.w_seed_principal > p.l_seed_principal
              WHEN p.w_seed_qualificatoria IS NOT NULL AND p.l_seed_qualificatoria IS NOT NULL
                THEN p.w_seed_qualificatoria > p.l_seed_qualificatoria
         END AS zebra
  FROM workspace.gold.fato_partida p
  JOIN workspace.gold.dim_fase f USING (id_fase)
  WHERE NOT p.flag_partida_incompleta
)
SELECT coalesce(fase, 'total')                          AS fase,
       count(*)                                         AS partidas,
       round(count(zebra) / count(*) * 100, 1)          AS pct_com_seed,
       round(avg(cast(zebra AS int)) * 100, 1)          AS pct_zebra,
       round(100 - avg(cast(zebra AS int)) * 100, 1)    AS pct_favorito
FROM base
GROUP BY ROLLUP(fase)
ORDER BY CASE fase WHEN 'qualificatoria' THEN 1 WHEN 'grupos' THEN 2 WHEN 'eliminatoria' THEN 3 ELSE 4 END

**Resultado.** O ranking prediz o resultado. O favorito vence 70,4% das partidas, contra os 50% que se esperaria se o seed não contribuísse com nada, e a fase não muda isso, com 70,5% na qualificatória, 70,7% nos grupos e 70,3% na eliminatória. A variação de 0,4 ponto percentual entre as fases está dentro do ruído, então a ideia de que a eliminatória seria mais imprevisível não se sustenta.

## 2. O jogo mudou ao longo das duas décadas? Duração das partidas e proporção de 2×1 por circuito e por período.

In [0]:
%sql
-- Só partidas completas e com duração plausível; 2×1 = três sets jogados
SELECT t.circuito,
       d.ciclo_olimpico,
       min(d.ano)                                            AS de,
       max(d.ano)                                            AS ate,
       count(*)                                              AS partidas,
       round(avg(p.duracao_min), 1)                          AS duracao_media_min,
       round(avg(cast(p.total_sets = 3 AS int)) * 100, 1)    AS pct_tres_sets,
       round(avg(p.total_pontos), 1)                         AS pontos_por_partida,
       round(avg(p.duracao_min) / avg(p.total_pontos) * 60, 1) AS segundos_por_ponto
FROM workspace.gold.fato_partida p
JOIN workspace.gold.dim_torneio t USING (id_torneio)
JOIN workspace.gold.dim_data d USING (id_data)
WHERE NOT p.flag_partida_incompleta
  AND NOT p.flag_duracao_suspeita
  AND p.duracao_min IS NOT NULL
GROUP BY t.circuito, d.ciclo_olimpico
ORDER BY t.circuito, min(d.ano)

**Resultado.** O jogo mudou, mas só em um dos circuitos.

- **FIVB:** a partida média encurtou de 45,4 minutos no ciclo de Pequim para 39,0 no de Tóquio, com 84 a 85 pontos por partida em todos os ciclos. O que caiu foi o tempo por ponto, de 32,2 para 27,8 segundos, enquanto a proporção de 2×1 subiu de 31,5% para cerca de 34%. Cada ponto passou a levar menos tempo, seja pelo rali ou pela pausa entre eles. As causas possíveis são ritmo de jogo, regras de intervalo ou critério de medição da duração, e o dado não distingue entre elas.
- **AVP:** nada se alterou. Duração entre 45 e 49 minutos e 2×1 entre 29% e 32% em todos os ciclos. O circuito não aparece em 2011 e 2012 porque faliu em agosto de 2010 e só voltou em 2012.
- **Comparação entre circuitos:** a diferença de duração era de 3 minutos no início da série e chega a 8 no fim.
- **Observação:** Sydney 2000 tem 0% de três sets e 104 segundos por ponto porque a partida era de set único até 15 com *side-out*, em que só quem sacava pontuava. O formato atual, de melhor de três sets a 21, entrou em 2001.

## 3. Quem é mais alto vence mais? A relação é a mesma no masculino e no feminino?

In [0]:
SELECT
  genero,
  FLOOR(f.altura_cm / 5) * 5                                     AS faixa_altura,
  COUNT(*)                                                       AS participacoes,
  ROUND(100 * AVG(CASE WHEN vencedor THEN 1 ELSE 0 END), 1)      AS taxa_vitoria
FROM workspace.gold.fato_atleta_partida f
JOIN workspace.gold.dim_atleta USING (id_atleta)
WHERE f.altura_cm IS NOT NULL
GROUP BY 1, 2
HAVING COUNT(*) >= 1000
ORDER BY 1, 2

**Resultado.** Sim, mas só nas pontas. No masculino, a taxa de vitória sobe com a altura: 37,9% na faixa
de 175 cm, 43,4% na de 180, e a partir daí um patamar de 50 a 51% entre 185 e 199 cm, onde estão três de
cada quatro participações. Acima de 200 cm ela volta a subir, para 56,1%, e chega a 59,2% na faixa de
205. Entre a faixa mais baixa e a mais alta são 21 pontos percentuais, a maior distância entre grupos de
atletas encontrada neste trabalho.

No feminino a forma é parecida, com o patamar mais largo: de 175 a 189 cm, onde estão 80% das
participações, a taxa fica entre 50,2 e 51,8%; a faixa de 170 vence 45,9% e a de 190 ou mais, 58,5%. A
exceção é a faixa de 165 cm, com 50,6%: são poucas atletas (5 mil participações) e, para estarem no
circuito com essa altura, provavelmente compensam de outra forma. A relação, portanto, é a mesma nos dois
gêneros na direção e na forma; no masculino a subida é mais longa porque a distribuição de alturas é mais
espalhada.

Na prática, altura não separa vencedores dentro da faixa comum do circuito, mas define quem tem
vantagem nas pontas: um homem acima de 2 metros ou uma mulher acima de 1,90 vence seis em cada dez
partidas, e abaixo de 1,80 (M) ou 1,75 (W) o atleta entra em desvantagem.

## 4. A idade pesa no resultado? Em que faixa etária a taxa de vitória é maior?

In [0]:
SELECT
  LEAST(GREATEST(FLOOR(idade_na_partida / 5) * 5, 15), 40)      AS faixa_etaria,
  COUNT(*)                                                      AS participacoes,
  ROUND(100 * AVG(CASE WHEN vencedor THEN 1 ELSE 0 END), 1)     AS taxa_vitoria
FROM workspace.gold.fato_atleta_partida
WHERE idade_na_partida IS NOT NULL
  AND NOT flag_idade_atipica
GROUP BY 1
ORDER BY 1

**Resultado.** Pesa, mas pouco depois dos 25. A taxa de vitória sobe com a idade até os 30 e só cai depois
dos 40. A faixa mais numerosa é a de 25 a 29 anos (114 mil participações, 37% do total), mas quem vence mais
é a de 30 a 39: 35% das participações e a maior taxa de vitória. A vantagem, porém, é pequena: 2 pontos
percentuais sobre a faixa de 25 a 29, e as duas ficam a menos de 3 pontos dos 50% que seria o esperado caso o
efeito da idade não impactasse na vitória. Essa vantagem pequena não é desprezível: medida sobre mais de 100 mil participações, não é ruído. Ainda
assim, ela é pequena perto da diferença entre os mais jovens e os demais: o menor de 20 anos vence 37% das
vezes, 15 pontos abaixo da faixa de 35 a 39, a maior distância entre grupos. Dessa forma, a idade não separa
vencedores entre os adultos; separa quem talvez ainda não tenha maturidade para o circuito.

## 5. Jogar em casa ajuda? Taxa de vitória quando o torneio é no país do atleta, comparada a fora.

In [0]:
SELECT
  circuito,
  flag_em_casa                                                   AS em_casa,
  COUNT(*)                                                       AS participacoes,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY circuito), 1) AS pct_participacoes,
  ROUND(100 * AVG(CASE WHEN vencedor THEN 1 ELSE 0 END), 1)      AS taxa_vitoria
FROM workspace.gold.fato_atleta_partida
JOIN workspace.gold.dim_torneio USING (id_torneio)
GROUP BY 1, 2
ORDER BY 1, 2 DESC

**Resultado.** Não. No AVP a pergunta quase não se aplica: 93,5% das participações são de atletas dos
Estados Unidos jogando nos Estados Unidos, e os dois lados vencem 50% (50,0 em casa, 50,7 fora). É no
FIVB, onde jogar em casa é a exceção (10,6% das participações), que a comparação vale, e o resultado é o
contrário do esperado: o atleta em casa vence 40,6% das partidas, contra 51,1% de quem joga fora. São 10,5
pontos percentuais contra o mandante.

Uma leitura possível é que o efeito venha da porta de entrada, não de nervosismo diante da torcida: o regulamento da FIVB
reserva vagas de convite para atletas do país-sede, que entram sem passar pelo ranking.

## 6. Nas partidas com estatística detalhada, qual fundamento mais separa vencedores de perdedores: ataque, saque, bloqueio ou defesa?

In [0]:
SELECT
  vencedor,
  COUNT(*)                        AS participacoes,
  ROUND(AVG(tot_attacks), 1)      AS ataques,
  ROUND(AVG(tot_kills), 1)        AS pontos_ataque,
  ROUND(AVG(tot_hitpct), 2)       AS aproveitamento,
  ROUND(AVG(tot_errors), 1)       AS erros_ataque,
  ROUND(AVG(tot_aces), 2)         AS aces,
  ROUND(AVG(tot_blocks), 2)       AS bloqueios,
  ROUND(AVG(tot_digs), 1)         AS defesas
FROM workspace.gold.fato_atleta_partida
WHERE tem_estatistica
  AND NOT flag_estatistica_invalida
GROUP BY vencedor
ORDER BY vencedor DESC

**Resultado.** O ataque. Vencedores e perdedores atacam o mesmo tanto (26,0 contra 26,9 tentativas por
partida), mas o vencedor converte mais (14,8 pontos contra 12,7) e erra menos (2,9 contra 4,4): o
aproveitamento vai de 0,31 para 0,48, e somando pontos ganhos e erros evitados são 3,5 pontos de diferença
por atleta em cada partida, mais que todos os outros fundamentos juntos. Bloqueio e ace têm a maior
diferença proporcional (1,73 contra 1,06 bloqueios; 1,32 contra 0,82 aces), mas acontecem pouco e somam
cerca de um ponto por atleta; defesa fica no meio (8,5 contra 7,2) e erro de saque quase não separa (2,0
contra 2,1). Na prática, o que mais separa quem vence não é atacar mais nem sacar melhor, e sim errar
menos no ataque.